In [1]:
from qdrant_client import QdrantClient

client = QdrantClient(
  host="localhost",
  prefer_grpc=True
)

info = client.info()
info

VersionInfo(title='qdrant - vector search engine', version='1.15.5', commit='48203e414e4e7f639a6d394fb6e4df695f808e51')

In [2]:
from qdrant_client.models import Distance, VectorParams

def ensure_collections():
  for name in ['video_frames', 'video_summary']:
    if not client.collection_exists(name):
      client.create_collection(
        name, 
        vectors_config=VectorParams(
          size=2560,
          distance=Distance.COSINE
        )
      )
      
ensure_collections()

In [3]:
from core.ai.embedding import Embedder

embedder = Embedder()

In [10]:
result1 = await embedder.get_embeddings("test video 1")
result2 = await embedder.get_embeddings("testify video")
result3 = await embedder.get_embeddings("tester video")
result4 = await embedder.get_embeddings("funny cats")

In [11]:
import numpy as np
import uuid
from qdrant_client.models import PointStruct

def upsert_video_summary(video_id: str, vector: np.ndarray):
  playload = { "video_id": video_id }
  
  client.upsert(
    collection_name="video_summary",
    points=[
      PointStruct(
        id=str(uuid.uuid4()),
        vector=vector.tolist(),
        payload=playload
      )
    ]
  )

upsert_video_summary("1", result1)
upsert_video_summary("2", result2)
upsert_video_summary("3", result3)
upsert_video_summary("4", result4)

In [22]:
async def search_similar_videos_by_text(query: str, top_k: int = 1):
  vec = await embedder.get_embeddings(query)
  
  points = client.query_points(collection_name="video_summary", query=vec, limit=top_k)
  return [
    {
        "payload": h.payload,
        "score": float(h.score),
    }
    for h in points.points
  ]
  
res = await search_similar_videos_by_text("tester video", top_k=10)
res

[{'payload': {'video_id': '3'}, 'score': 1.0000001192092896},
 {'payload': {'video_id': '1'}, 'score': 0.8433226346969604},
 {'payload': {'video_id': '2'}, 'score': 0.8162723183631897},
 {'payload': {'video_id': '4'}, 'score': 0.4913404583930969}]